# Broken Building Reporter — YOLO11 Model Fine-Tuning & Training
This notebook downloads the 3 datasets from Dharma's Roboflow Workspace (`dharmas-workspace-wa1qs`), unifies the labels into a single 3-class dataset (`crack`, `water_leakage`, `exposed_wire`), and fine-tunes a high-accuracy YOLO11 model using a free GPU.

**Hardware requirement:** Runtime -> Change runtime type -> Hardware accelerator: **T4 GPU**.

In [ ]:
# Step 1: Install Ultralytics and Roboflow
!pip install -q ultralytics roboflow

In [ ]:
# Step 2: Download the 3 datasets from Dharma's Roboflow Workspace
from roboflow import Roboflow
import os, shutil
from pathlib import Path
import yaml

# Enter your Roboflow API key (get it from: https://app.roboflow.com/settings/api)
ROBOFLOW_API_KEY = input("Enter your Roboflow API Key: ").strip()

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
workspace = rf.workspace("dharmas-workspace-wa1qs")

base_dir = Path("/content/dataset_pipeline")
raw_dir = base_dir / "raw"
merged_dir = Path("/content/final_building_dataset")

for split in ["train", "valid", "test"]:
    (merged_dir / "images" / split).mkdir(parents=True, exist_ok=True)
    (merged_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

configs = [
    {"project": "building-crack-detection-dmhfv-as5ew", "version": 1, "id": 0, "name": "crack"},
    {"project": "water-leakage-yjjbh-ldonl", "version": 1, "id": 1, "name": "water_leakage"},
    {"project": "wires-quxig-ttb28-rmqn1", "version": 1, "id": 2, "name": "exposed_wire"}
]

for cfg in configs:
    print(f"Downloading {cfg['name']} ({cfg['project']})...")
    proj = workspace.project(cfg['project'])
    ver = proj.version(cfg['version'])
    dl = ver.download("yolov11", location=str(raw_dir / cfg['project']))
    
    # Copy and re-map class indices
    for split in ["train", "valid", "test"]:
        src_img = Path(dl.location) / split / "images"
        src_lbl = Path(dl.location) / split / "labels"
        if not src_img.exists():
            src_img = Path(dl.location) / split
            src_lbl = Path(dl.location) / split
        if not src_img.exists():
            continue
            
        for img in src_img.glob("*.*"):
            if img.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
                continue
            dest_name = f"{cfg['name']}_{img.stem}"
            shutil.copy2(img, merged_dir / "images" / split / f"{dest_name}{img.suffix}")
            
            lbl = src_lbl / f"{img.stem}.txt"
            dest_lbl = merged_dir / "labels" / split / f"{dest_name}.txt"
            if lbl.exists():
                with open(lbl, "r") as f:
                    lines = f.readlines()
                remapped = []
                for line in lines:
                    p = line.strip().split()
                    if len(p) >= 5:
                        p[0] = str(cfg['id'])
                        remapped.append(" ".join(p))
                with open(dest_lbl, "w") as f:
                    f.write("\n".join(remapped) + "\n")

# Create data.yaml
yaml_content = {
    "path": "/content/final_building_dataset",
    "train": "images/train",
    "val": "images/valid",
    "test": "images/test",
    "names": {0: "crack", 1: "water_leakage", 2: "exposed_wire"}
}
with open(merged_dir / "data.yaml", "w") as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print("Unified dataset prepared successfully!")

In [ ]:
# Step 3: Train YOLO11 Model for 80 Epochs with AdamW and Cosine LR
from ultralytics import YOLO

# Using yolo11s (Small - 9.4M params) for superior accuracy over yolo11n (Nano)
model = YOLO("yolo11s.pt")

results = model.train(
    data="/content/final_building_dataset/data.yaml",
    epochs=80,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.002,
    cos_lr=True,
    patience=20,
    save=True,
    verbose=True
)

In [ ]:
# Step 4: Validate and Plot Performance Metrics
metrics = model.val()
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

# Step 5: Download trained best.pt
from google.colab import files
best_weights = Path(model.trainer.save_dir) / "weights" / "best.pt"
print(f"Downloading {best_weights}...")
files.download(str(best_weights))